# ZJU Quantum Compiler — V14 Demo (AI vs SABRE)

本 notebook 展示用 V14.2 的 RL 路由器 (`models/v14_tokyo20/v7_ibm_tokyo_best.pt`) 编译典型量子电路，并与 Qiskit 默认的 SABRE 算法对比 SWAP 数。

**项目仓库**: https://github.com/qqyyqq812/ZJU-Quantum-Compiler

---

## 环境准备

```bash
# 在项目根目录
pip install -e .
```

Notebook 假定从项目根目录启动 Jupyter（`cd /path/to/ZJU-Quantum-Compiler && jupyter notebook`）。

In [ ]:
import sys
from pathlib import Path

# Notebook 在 notebooks/ 下；把项目根加入 sys.path
PROJECT_ROOT = Path('.').resolve().parent if Path('src').exists() == False else Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from qiskit import QuantumCircuit, transpile
from src.benchmarks.circuits import generate_qft, generate_qaoa, generate_grover, generate_random
from src.benchmarks.topologies import get_topology
from src.compiler.pass_manager import AIRouter

import time, numpy as np, matplotlib.pyplot as plt
import pandas as pd

print(f'Project root: {PROJECT_ROOT}')

## 1. 加载训练好的 V14 路由器 + IBM Tokyo 20Q 拓扑

In [ ]:
MODEL_PATH = PROJECT_ROOT / 'models' / 'v14_tokyo20' / 'v7_ibm_tokyo_best.pt'
TOPOLOGY = 'ibm_tokyo'

cm = get_topology(TOPOLOGY)
print(f'Topology: {TOPOLOGY} — {cm.size()} 物理比特, {len(cm.get_edges())} 条耦合边')

if MODEL_PATH.is_file():
    router = AIRouter(cm, model_path=str(MODEL_PATH))
    print(f'AI 路由器已加载: {MODEL_PATH.name}')
else:
    print(f'WARN: 未找到模型 {MODEL_PATH}，将以随机策略 fallback 演示')
    router = AIRouter(cm, model_path=None)

## 2. 测试电路集（覆盖 5Q）

这些电路在 V14.2 课程学习的 Stage 0-2 上已收敛（ep25333）。20Q 大电路（Stage 3-4）目前在 V15 (MCTS+GNN) 路线下推进，参见 `docs/technical/decisions.md §V15`。

In [ ]:
BENCHMARKS = [
    ('QFT-3', lambda: generate_qft(3)),
    ('QFT-5', lambda: generate_qft(5)),
    ('GHZ-5', lambda: QuantumCircuit.from_qasm_str('OPENQASM 2.0; include "qelib1.inc"; qreg q[5]; h q[0]; cx q[0],q[1]; cx q[1],q[2]; cx q[2],q[3]; cx q[3],q[4];')),
    ('QAOA-5', lambda: generate_qaoa(5, p=1)),
    ('Grover-3', lambda: generate_grover(3)),
    ('Random-5', lambda: generate_random(5, depth=10, seed=42)),
]

for name, build in BENCHMARKS:
    qc = build()
    print(f'  {name:12s}: {qc.num_qubits} qubits, {qc.size()} gates, depth={qc.depth()}')

## 3. 双路对比：AI Router vs Qiskit SABRE

In [ ]:
rows = []
for name, build in BENCHMARKS:
    qc = build()
    # AI 路由
    t0 = time.time()
    ai_out = router.route_count_only(qc, max_steps=2000)
    ai_dt = (time.time() - t0) * 1000
    # SABRE 基线
    t0 = time.time()
    sabre_qc = transpile(qc, coupling_map=cm, optimization_level=1, routing_method='sabre', seed_transpiler=42)
    sabre_dt = (time.time() - t0) * 1000
    sabre_swaps = sabre_qc.count_ops().get('swap', 0)
    rel = (ai_out['ai_swaps'] - sabre_swaps) / max(sabre_swaps, 1)
    rows.append({
        'circuit': name,
        'qubits': qc.num_qubits,
        'gates': qc.size(),
        'AI SWAP': ai_out['ai_swaps'],
        'AI time (ms)': round(ai_dt, 1),
        'SABRE SWAP': sabre_swaps,
        'SABRE time (ms)': round(sabre_dt, 1),
        'AI/SABRE': f'{rel*100:+.0f}%',
        'AI done': ai_out['done'],
    })
df = pd.DataFrame(rows)
df

## 4. 可视化

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 5))
x = np.arange(len(df))
w = 0.35
ax.bar(x - w/2, df['AI SWAP'], w, label='V14.2 AI Router', color='#2E86AB')
ax.bar(x + w/2, df['SABRE SWAP'], w, label='Qiskit SABRE', color='#F4A261')
ax.set_xticks(x)
ax.set_xticklabels(df['circuit'], rotation=20)
ax.set_ylabel('SWAP count (lower is better)')
ax.set_title(f'V14.2 AI Router vs Qiskit SABRE ({TOPOLOGY})')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. 训练历史曲线（来自 V14 真实训练数据）

In [ ]:
import json
hist_path = PROJECT_ROOT / 'models' / 'v14_tokyo20' / 'history_v7_ibm_tokyo.json'
if hist_path.exists():
    h = json.load(open(hist_path))
    swaps = np.array(h['episode_swaps'])
    stages = np.array(h['curriculum_stages'])
    window = 200
    smoothed = np.convolve(swaps, np.ones(window)/window, mode='valid')
    fig, ax = plt.subplots(1, 1, figsize=(12, 5))
    ax.plot(swaps, alpha=0.2, label='raw episode SWAP')
    ax.plot(np.arange(window-1, len(swaps)), smoothed, label=f'rolling avg (w={window})', linewidth=2, color='red')
    # 课程阶段背景
    stage_colors = ['#FFE5E5', '#E5F0FF', '#E5FFE5', '#FFF5E5', '#F0E5FF']
    prev = stages[0]; start = 0
    for i, s in enumerate(stages):
        if s != prev or i == len(stages)-1:
            ax.axvspan(start, i, alpha=0.3, color=stage_colors[int(prev) % 5])
            prev = s; start = i
    ax.set_xlabel('Episode')
    ax.set_ylabel('Total SWAPs')
    ax.set_title('V14.2 训练曲线 (4616 episodes，背景色 = 课程阶段)')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
    print(f'Total episodes: {len(swaps)}, Final stage: {int(stages[-1])}')
else:
    print('history.json 未找到')

## 6. 总结 + V15 路线图

### V14.2 现状
- ✅ Stage 0-2 (3-5Q) 完整可用，AI SWAP 通常 ≤ 1.2× SABRE
- ⚠️ Stage 3 (10Q) 在 PPO 下卡住（reward landscape 平坦）
- 详见 `docs/technical/decisions.md §V14.1` 踩坑记录

### V15 路线（开发中）
- 算法切换：PPO → MCTS+GNN（AlphaZero 风格）
- 公开 SOTA 参考：QRoute (AAAI 2022)、AlphaRouter (Amazon 2024)
- 保留 70% V14 工程：env / GNN / SABRE 缓存 / curriculum 全部继承
- 详见 `docs/technical/decisions.md §V15`

### 复现
```bash
# 跑评测
python scripts/eval_v14_vs_sabre.py --model models/v14_tokyo20/v7_ibm_tokyo_best.pt
# 跑 MQT-Bench
python scripts/eval_mqt_bench.py
```